# 04 · Comparative Summary

This notebook synthesises results from notebooks 02 and 03 into a single cross-metric
comparison suitable for the thesis results chapter.

**Units:**
- Energy → **J** (Joules)
- Time → **s** (seconds)
- EDP → **J·s** (Joule-seconds)

**Outputs:**
- `outputs/ranking_summary.csv` — full multi-metric ranking table
- Radar chart per paradigm
- Normalised metrics heatmap
- Top 3 / Bottom 3 per metric
- Key findings bullet list

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from itertools import combinations
from pathlib import Path
%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 150, 'figure.figsize': (12, 6)})

In [ ]:
# ── Column names as they appear in results_clean_runs.csv ────────────────────
# Units already converted by notebooks/01_data_cleaning.ipynb
COL_CPU_ENERGY = 'cpu_energy_rapl_msr_component-package_0-j'
COL_MEM_ENERGY = 'memory_energy_rapl_msr_component-dram_0-j'
COL_TIME       = 'phase_time_syscall_system-system-s'
COL_CPU_CARBON = 'cpu_carbon_rapl_msr_component-package_0-g'
COL_MEM_CARBON = 'memory_carbon_rapl_msr_component-dram_0-g'

ALPHA = 0.05

FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR = Path('outputs')
OUTPUTS_DIR.mkdir(exist_ok=True)

LANG_DISPLAY = {
    'c': 'C', 'cpp': 'C++', 'csharp': 'C#', 'fsharp': 'F#',
    'nodejs': 'JavaScript', 'dart': 'Dart', 'erlang': 'Erlang',
    'go': 'Go', 'haskell': 'Haskell', 'java': 'Java', 'lua': 'Lua',
    'ocaml': 'OCaml', 'perl': 'Perl', 'php': 'PHP',
    'python': 'Python', 'ruby': 'Ruby', 'rust': 'Rust', 'swift': 'Swift',
}

PARADIGM = {
    'C': 'AOT', 'C++': 'AOT', 'C#': 'AOT', 'Dart': 'AOT', 'Go': 'AOT',
    'Haskell': 'AOT', 'Java': 'AOT', 'OCaml': 'AOT', 'Rust': 'AOT', 'Swift': 'AOT',
    'Erlang': 'JIT', 'F#': 'JIT', 'JavaScript': 'JIT', 'PHP': 'JIT', 'Ruby': 'JIT',
    'Lua': 'Interpreted', 'Perl': 'Interpreted', 'Python': 'Interpreted',
}

PARADIGM_COLORS = {'AOT': '#2980b9', 'JIT': '#e67e22', 'Interpreted': '#27ae60'}
PARADIGM_ORDER  = ['AOT', 'JIT', 'Interpreted']

# Data pre-cleaned by notebooks/01_data_cleaning.ipynb:
#   - Outliers removed per (language × benchmark) group, IQR fence on CPU energy + time
#   - Units already converted (J, s, g, MB, W)
df = pd.read_csv('../../results/results_clean_runs.csv')
df['language'] = df['language'].replace(LANG_DISPLAY)
df['paradigm'] = df['language'].map(PARADIGM)

print(f"Shape: {df.shape}")
print(f"Languages ({df['language'].nunique()}): {sorted(df['language'].unique())}")
print(f"Benchmarks ({df['benchmark'].nunique()}): {sorted(df['benchmark'].unique())}")
print("Units: energy=J | time=s | carbon=g | disk/net=MB | power=W")
df.head(3)

## 1. Multi-Metric Ranking Table

Each language is ranked (1 = best) on four metrics:
- Mean CPU Energy (J)
- Mean Memory Energy (J)
- Mean Execution Time (s)
- Mean EDP — (CPU + Memory Energy) × Time (J·s)

Values use benchmark-level means averaged across all 8 benchmarks (equal benchmark weight).
The overall rank is the average of the four individual ranks.

In [ ]:
df['EDP'] = (df[COL_CPU_ENERGY] + df[COL_MEM_ENERGY]) * df[COL_TIME]  # J · s

# Two-step aggregation: benchmark-level mean → language-level mean (equal benchmark weight)
per_bench = df.groupby(['language', 'benchmark']).agg(
    paradigm    = ('paradigm', 'first'),
    cpu_mean_J  = (COL_CPU_ENERGY, 'mean'),
    mem_mean_J  = (COL_MEM_ENERGY, 'mean'),
    time_mean_s = (COL_TIME, 'mean'),
    edp_mean_Js = ('EDP', 'mean'),
)
agg = per_bench.groupby('language').agg(
    paradigm    = ('paradigm', 'first'),
    cpu_mean_J  = ('cpu_mean_J', 'mean'),
    mem_mean_J  = ('mem_mean_J', 'mean'),
    time_mean_s = ('time_mean_s', 'mean'),
    edp_mean_Js = ('edp_mean_Js', 'mean'),
).round(4)

agg['cpu_rank']  = agg['cpu_mean_J'].rank().astype(int)
agg['mem_rank']  = agg['mem_mean_J'].rank().astype(int)
agg['time_rank'] = agg['time_mean_s'].rank().astype(int)
agg['edp_rank']  = agg['edp_mean_Js'].rank().astype(int)
agg['overall_rank'] = (
    (agg['cpu_rank'] + agg['mem_rank'] + agg['time_rank'] + agg['edp_rank']) / 4
).round(2)

ranking = agg.sort_values('edp_rank')
ranking.index.name = 'Language'
ranking[['paradigm', 'cpu_mean_J', 'mem_mean_J', 'time_mean_s', 'edp_mean_Js',
         'cpu_rank', 'mem_rank', 'time_rank', 'edp_rank', 'overall_rank']]

In [ ]:
export = ranking[['paradigm', 'cpu_mean_J', 'mem_mean_J', 'time_mean_s',
                  'edp_mean_Js', 'cpu_rank', 'mem_rank', 'time_rank',
                  'edp_rank', 'overall_rank']].copy()
export.columns = ['Paradigm', 'CPU Energy (J)', 'Mem Energy (J)', 'Time (s)',
                  'EDP (J·s)', 'CPU Rank', 'Mem Rank', 'Time Rank', 'EDP Rank',
                  'Overall Rank']
export.to_csv(OUTPUTS_DIR / 'ranking_summary.csv')
print(f"Saved → {OUTPUTS_DIR / 'ranking_summary.csv'}")
export

## 2. Normalized Efficiency Comparison

Each metric is expressed as a **ratio relative to the best (lowest) language** — so 1.0 = best
and e.g. 5.0 means this language consumes 5× more CPU energy / RAM energy / time than the
most efficient language. Values are computed from the **mean** across all 8 benchmarks,
matching the CLBG presentation style.

In [ ]:
# Aggregate to per-(language × benchmark) means first, then average across benchmarks.
# This gives each benchmark equal weight regardless of how many runs survived outlier removal.
per_bench = df.groupby(['language', 'benchmark'])[[COL_CPU_ENERGY, COL_MEM_ENERGY, COL_TIME]].mean()
norm_agg  = per_bench.groupby('language').mean()

norm_agg['CPU Energy Normalized'] = (norm_agg[COL_CPU_ENERGY] / norm_agg[COL_CPU_ENERGY].min()).round(2)
norm_agg['RAM Energy Normalized'] = (norm_agg[COL_MEM_ENERGY] / norm_agg[COL_MEM_ENERGY].min()).round(2)
norm_agg['Time Normalized']       = (norm_agg[COL_TIME]        / norm_agg[COL_TIME].min()).round(2)

norm_ranking = (
    norm_agg[['CPU Energy Normalized', 'RAM Energy Normalized', 'Time Normalized']]
    .sort_values('CPU Energy Normalized')
    .reset_index()
    .rename(columns={'language': 'Language'})
)

norm_ranking.index = norm_ranking.index + 1  # rank starts at 1
norm_ranking.index.name = 'Rank'

norm_ranking

## 3. Radar Chart — Paradigm Profile

One polygon per paradigm, with axes normalised to [0, 1] (0 = best, 1 = worst).
This reveals the energy/speed trade-off profile of each execution model.

In [ ]:
paradigm_agg = df.groupby('paradigm').agg(
    cpu  = (COL_CPU_ENERGY, 'median'),
    mem  = (COL_MEM_ENERGY, 'median'),
    time = (COL_TIME, 'median'),
    edp  = ('EDP', 'median'),
)

norm = (paradigm_agg - paradigm_agg.min()) / (paradigm_agg.max() - paradigm_agg.min())

categories = ['CPU Energy (J)', 'Memory Energy (J)', 'Execution Time (s)', 'EDP (J·s)']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})
for paradigm in PARADIGM_ORDER:
    values = norm.loc[paradigm].tolist()
    values += values[:1]
    ax.plot(angles, values, color=PARADIGM_COLORS[paradigm], linewidth=2, label=paradigm)
    ax.fill(angles, values, color=PARADIGM_COLORS[paradigm], alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10)
ax.set_ylim(0, 1)
ax.set_title('Paradigm Profile (normalised; lower = better)', fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'paradigm_radar.png', bbox_inches='tight')
plt.show()

## 4. Normalised Metrics Heatmap

All four metrics normalised to [0, 1] per column. Lower values (greener) are better.
Enables direct visual comparison of language profiles across all metrics in human-readable units.

In [ ]:
metrics = {
    'CPU Energy (J)': COL_CPU_ENERGY,
    'Mem Energy (J)': COL_MEM_ENERGY,
    'Time (s)':       COL_TIME,
    'EDP (J·s)':      'EDP',
}
norm_df = pd.DataFrame(index=ranking.index)
for label, col in metrics.items():
    vals = df.groupby('language')[col].median()
    norm_df[label] = (vals - vals.min()) / (vals.max() - vals.min())

norm_df = norm_df.loc[ranking.index]  # order by overall rank

fig, ax = plt.subplots(figsize=(9, 10))
sns.heatmap(norm_df, annot=True, fmt='.2f',
            cmap=sns.diverging_palette(120, 10, as_cmap=True),
            center=0.5, vmin=0, vmax=1, ax=ax,
            linewidths=0.5, cbar_kws={'label': '0 = best, 1 = worst'})
ax.set_title('Normalised Metric Heatmap (sorted by Overall Rank)', fontsize=12)
ax.set_ylabel('Language (best → worst overall)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'normalised_heatmap.png', bbox_inches='tight')
plt.show()

## 5. Top 3 / Bottom 3 per Metric

Quick-reference tables for the most and least efficient languages on each metric.
All values in human-readable units (J, s, J·s).

In [ ]:
metric_cols = {
    'CPU Energy (J)':    COL_CPU_ENERGY,
    'Memory Energy (J)': COL_MEM_ENERGY,
    'Execution Time (s)':COL_TIME,
    'EDP (J·s)':         'EDP',
}
units = {'CPU Energy (J)': 'J', 'Memory Energy (J)': 'J',
         'Execution Time (s)': 's', 'EDP (J·s)': 'J·s'}

for label, col in metric_cols.items():
    med_series = df.groupby('language')[col].median().sort_values()
    top3 = med_series.head(3)
    bot3 = med_series.tail(3)
    unit = units[label]
    print(f"\n{'─'*55}")
    print(f"  {label}")
    print(f"  Top 3 (most efficient):")
    for lang, val in top3.items():
        print(f"    • {lang:12s} ({PARADIGM[lang]:12s})  {val:>10.3f} {unit}")
    print(f"  Bottom 3 (least efficient):")
    for lang, val in bot3.items():
        print(f"    • {lang:12s} ({PARADIGM[lang]:12s})  {val:>10.3f} {unit}")

## 6. Key Findings

A bullet-point summary of the main findings in human-readable units (J, s, J·s),
suitable for direct citation in the thesis.

In [ ]:
cpu_rank_ser  = df.groupby('language')[COL_CPU_ENERGY].median().sort_values()
time_rank_ser = df.groupby('language')[COL_TIME].median().sort_values()
edp_rank_ser  = df.groupby('language')['EDP'].median().sort_values()

aot_cpu  = df[df['paradigm']=='AOT'][COL_CPU_ENERGY].median()
jit_cpu  = df[df['paradigm']=='JIT'][COL_CPU_ENERGY].median()
int_cpu  = df[df['paradigm']=='Interpreted'][COL_CPU_ENERGY].median()
aot_time = df[df['paradigm']=='AOT'][COL_TIME].median()
jit_time = df[df['paradigm']=='JIT'][COL_TIME].median()
int_time = df[df['paradigm']=='Interpreted'][COL_TIME].median()

print("""
KEY FINDINGS — Benchmark Energy & Time Analysis (18 languages, 8 CLBG benchmarks)
═════════════════════════════════════════════════════════════════════════════════""")

print(f"""
CPU ENERGY (unit: J)
  • Most efficient:   {', '.join(cpu_rank_ser.head(3).index)}
  • Least efficient:  {', '.join(cpu_rank_ser.tail(3).index)}
  • AOT median:       {aot_cpu:.2f} J
  • JIT median:       {jit_cpu:.2f} J  ({jit_cpu/aot_cpu:.1f}× AOT)
  • Interpreted med.: {int_cpu:.2f} J  ({int_cpu/aot_cpu:.1f}× AOT)

EXECUTION TIME (unit: s)
  • Fastest:          {', '.join(time_rank_ser.head(3).index)}
  • Slowest:          {', '.join(time_rank_ser.tail(3).index)}
  • AOT median:       {aot_time:.2f} s
  • JIT median:       {jit_time:.2f} s  ({jit_time/aot_time:.1f}× AOT)
  • Interpreted med.: {int_time:.2f} s  ({int_time/aot_time:.1f}× AOT)

ENERGY-DELAY PRODUCT (unit: J·s)
  • Best EDP:         {', '.join(edp_rank_ser.head(3).index)}
  • Worst EDP:        {', '.join(edp_rank_ser.tail(3).index)}
  • Best median EDP:  {edp_rank_ser.iloc[0]:.2f} J·s  ({edp_rank_ser.index[0]})
  • Worst median EDP: {edp_rank_ser.iloc[-1]:.2f} J·s ({edp_rank_ser.index[-1]})

Note: All values are medians across 8 CLBG benchmarks and ~10 runs per combination.
Use non-parametric tests (notebooks 02–03) when comparing paradigm distributions.
""")